# Train A-Geo or C-Geo on G1_smooth_v1
Order: **1 bootstrap/dependencies →2 scientific environment →3 experiment configuration →4 compatible numerical certification →5 plan review →6 explicit launch**. Fresh Colab setup retains consistent preloaded numerical binaries; no manual or automatic restart is used. A new environment is numerically qualified before training. Run All never launches an optimizer.


In [1]:
# Setup configuration — used only in Colab. Local sessions need an installed project.
REPO_URL = "SET_YOUR_GITHUB_REPOSITORY_URL"
REPO_REF = "main"  # Prefer an immutable full commit SHA or release tag for research.
PROJECT_DIR = "/content/pde-inverse-research"

USE_GOOGLE_DRIVE = False
DRIVE_ROOT = "/content/drive/MyDrive/PDEInverseResearch"
LOCAL_OUTPUT_DIR = "/content/pde-inverse-outputs"

# Geometry installation extra required for G1_smooth_v1.
ENABLE_GEOMETRY = True


In [2]:
# Automatic setup — advanced implementation (collapsed). Run this cell normally.
# Stdlib bootstrap is embedded because pdeep is absent in a fresh Colab runtime.
import sys, json
from pathlib import Path
try:
    from IPython import get_ipython
    _shell = get_ipython()
except ImportError:
    _shell = None
IS_COLAB = "google.colab" in sys.modules or type(_shell).__module__.startswith("google.colab")

def ensure_checkout(repo_url, repo_ref, project_dir, *, git_env=None):
    """Clone only absent destinations; existing worktrees are strictly read-only.

    Supports token-free HTTPS and Git-over-SSH URLs. External credential helpers
    or SSH agents handle private authentication. Raw Git output stays in memory.
    """
    import os, re, subprocess, tempfile
    from pathlib import Path
    from urllib.parse import urlsplit
    import sys
    if sys.version_info < (3, 10):
        raise RuntimeError('PDEep requires Python >=3.10; select a supported runtime before setup.')
    def identity(value):
        if not isinstance(value,str) or not value or any(c in value for c in '\n\r\0'):
            raise ValueError('Configure a token-free repository URL; credentials must not be embedded.')
        if value.startswith('git@'):
            match=re.fullmatch(r'git@([A-Za-z0-9.-]+):([A-Za-z0-9_./-]+)',value)
            if not match:raise ValueError('Use git@HOST:OWNER/REPO.git with an external SSH agent.')
            host,path=match.groups()
        else:
            url=urlsplit(value)
            if url.scheme not in {'https','ssh'} or not url.hostname or url.password or url.query or url.fragment:
                raise ValueError('Use token-free HTTPS or Git SSH; credential URLs, queries and fragments are rejected.')
            if (url.scheme=='https' and url.username is not None) or (url.scheme=='ssh' and url.username not in {None,'git'}):
                raise ValueError('Do not put credentials/user tokens in repository URLs; use an external credential helper or SSH agent.')
            host=url.hostname;path=url.path.lstrip('/')
            if url.port not in {None,22,443}:raise ValueError('Unsupported repository port.')
        path=path.rstrip('/').removesuffix('.git')
        if not path or len(path.split('/'))<2 or '..' in path.split('/'):
            raise ValueError('Configure REPO_URL with OWNER/REPO; no local or credential-bearing URL is accepted.')
        if host.lower()=='github.com':path=path.lower()
        return host.lower()+'/'+path
    expected=identity(repo_url)
    if not isinstance(repo_ref,str) or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_./-]*',repo_ref) or '..' in repo_ref:
        raise ValueError('REPO_REF must be a branch, tag or full commit SHA, not a command or revision expression.')
    destination=Path(project_dir).expanduser().resolve()
    env=dict(os.environ);env.update(git_env or {});env['GIT_TERMINAL_PROMPT']='0'
    def git(arguments, *, cwd=None, purpose='repository verification'):
        try:result=subprocess.run(['git',*arguments],cwd=cwd,env=env,capture_output=True,text=True,timeout=180)
        except FileNotFoundError:raise RuntimeError('Git is missing; install Git before repository setup.') from None
        except (OSError,subprocess.TimeoutExpired):raise RuntimeError('Git '+purpose+' could not complete; no credentials/output were logged.') from None
        if result.returncode:
            error=result.stderr.lower()
            if any(s in error for s in ('authentication','could not read username','publickey','permission denied')):
                raise RuntimeError('Git authentication failed. Configure an external credential helper or SSH agent with read access; do not put secrets in this notebook.')
            raise RuntimeError('Git '+purpose+' failed. Check repository existence, access and REPO_REF. Raw output is suppressed to protect credentials.')
        return result.stdout.strip()
    def head(path):return git(['rev-parse','--verify','HEAD^{commit}'],cwd=path)
    def resolved_ref(path):
        if re.fullmatch(r'[0-9a-fA-F]{40}|[0-9a-fA-F]{64}',repo_ref):wanted=repo_ref.lower()
        else:
            refs=[repo_ref,repo_ref+'^{}'] if repo_ref.startswith('refs/') else ['refs/heads/'+repo_ref,'refs/tags/'+repo_ref,'refs/tags/'+repo_ref+'^{}']
            lines=git(['ls-remote','--exit-code','origin',*refs],cwd=path,purpose='remote reference lookup')
            mapping=dict((ref,sha) for sha,ref in (line.split() for line in lines.splitlines()))
            if repo_ref.startswith('refs/'):wanted=mapping.get(repo_ref+'^{}',mapping.get(repo_ref))
            else:
                branch=mapping.get('refs/heads/'+repo_ref);tag=mapping.get('refs/tags/'+repo_ref+'^{}',mapping.get('refs/tags/'+repo_ref))
                if branch and tag and branch!=tag:raise ValueError('Branch/tag name is ambiguous; use refs/heads/... or refs/tags/....')
                wanted=branch or tag
            if not wanted:raise ValueError('Configured REPO_REF was not found on origin.')
        return wanted
    existed=destination.exists()
    if existed:
        if not destination.is_dir() or not (destination/'.git').exists():
            raise ValueError('PROJECT_DIR exists but is not a Git checkout; choose a separate empty directory. Nothing was overwritten.')
        top=Path(git(['rev-parse','--show-toplevel'],cwd=destination)).resolve()
        if top!=destination:raise ValueError('PROJECT_DIR must be the repository root, not a nested directory.')
        origin=git(['config','--get','remote.origin.url'],cwd=destination)
        if identity(origin)!=expected:raise ValueError('Existing checkout origin differs from REPO_URL. No files or remotes were changed.')
        if git(['status','--porcelain','--untracked-files=normal'],cwd=destination):
            raise ValueError('Existing checkout has local modifications/untracked files. Commit them yourself or choose a new PROJECT_DIR; no reset/checkout was performed.')
        commit=head(destination)
        wanted=resolved_ref(destination)
        if commit!=wanted:
            raise ValueError('Existing checkout is at '+commit+' but REPO_REF resolves to '+wanted+'. Choose a new PROJECT_DIR or update it explicitly; nothing was reset.')
    else:
        destination.parent.mkdir(parents=True,exist_ok=True)
        # A failed clone stays in its own scratch directory, not PROJECT_DIR.
        with tempfile.TemporaryDirectory(prefix='.pdeep-clone-',dir=destination.parent) as scratch:
            checkout=Path(scratch)/'checkout'
            git(['clone','--no-checkout','--',repo_url,str(checkout)],purpose='clone')
            wanted=resolved_ref(checkout)
            git(['fetch','--depth','1','origin',wanted],cwd=checkout,purpose='configured reference fetch')
            git(['checkout','--detach','FETCH_HEAD'],cwd=checkout,purpose='fresh checkout')
            commit=head(checkout)
            if commit!=wanted:raise RuntimeError('Fetched checkout differs from the resolved reference; setup stopped.')
            required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
            if any(not (checkout/p).is_file() for p in required):
                raise ValueError('Repository/ref lacks the expected PDEep structure or this Colab setup version; select a compatible commit.')
            os.rename(checkout,destination)
    required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
    if any(not (destination/p).is_file() for p in required):raise ValueError('Expected pyproject.toml, pdeep package/setup and scientific contract are missing.')
    return {'project_dir':str(destination),'repository_identity':expected,'requested_ref':repo_ref,'git_commit':commit,
            'existing_checkout':existed,'worktree_modified_by_setup':not existed,'credentials_recorded':False}

BOOTSTRAP = None
if IS_COLAB:
    print("1. Verifying repository and preparing dependencies before scientific project imports.")
    BOOTSTRAP = ensure_checkout(REPO_URL, REPO_REF, PROJECT_DIR)
    import runpy
    _reference_setup = runpy.run_path(str(Path(BOOTSTRAP["project_dir"]) / "src/pdeep/smooth/bootstrap.py"))
    ENVIRONMENT_SETUP = _reference_setup["prepare_reference_environment"](BOOTSTRAP["project_dir"], is_colab=True)
    _reference_setup["install_project"](BOOTSTRAP["project_dir"], source_revision=BOOTSTRAP["git_commit"])
    print("Dependency preparation complete; no loaded binary was replaced. Numerical qualification follows configuration.")
print("2. Scientific imports and resolved environment.")
from pdeep.colab_setup import finish_setup
SETUP = finish_setup(is_colab=IS_COLAB, bootstrap=BOOTSTRAP, use_drive=USE_GOOGLE_DRIVE,
    drive_root=DRIVE_ROOT, local_output_dir=LOCAL_OUTPUT_DIR)
SETUP_CONFIGURATION = (REPO_URL, REPO_REF, PROJECT_DIR, USE_GOOGLE_DRIVE, DRIVE_ROOT, LOCAL_OUTPUT_DIR)
ROOT = Path(SETUP["project_dir"])
from IPython.display import display, Markdown
from pdeep.notebook_workflows import initialize
from pdeep.runtime import capabilities
print("Colab:", IS_COLAB, "| imported source:", ROOT / "src/pdeep")
print("Exact Git commit:", SETUP["git_commit"] or "unavailable: local project is not a Git checkout")
print("Python:", SETUP["runtime"]["python"])
print("Dependency versions:", json.dumps(SETUP["runtime"]["dependencies"], indent=2))
for c in capabilities(): print(c.device, "available="+str(c.available), c.detail)
if IS_COLAB: print("Validated CUDA:", SETUP["cuda"])
print("Local smooth run root:", Path(SETUP["local_output_dir"]) / "G1_smooth_v1/runs")
print("Drive checkpoints:", None if not SETUP["drive_root"] else Path(SETUP["drive_root"]) / "checkpoints")
print(SETUP["message"])

from pdeep.smooth.bootstrap import environment_versions
print("Python executable:", sys.executable)
print("Installed / loaded dependencies:", json.dumps(environment_versions(), sort_keys=True))


Colab: False | imported source: /Users/nicolasackermann/Documents/ChatGPT/PDEep/src/pdeep
Exact Git commit: bcf76427e542c4f612277f1e83265c8874271e23
Python: 3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:47) [Clang 13.0.0 (clang-1300.0.29.30)]
Dependency versions: {
  "pdeep-research": "0.1.0",
  "torch": "2.15.0",
  "numpy": "2.5.4",
  "scipy": "1.18.1",
  "matplotlib": "3.11.2",
  "pytest": "9.2.0",
  "notebook": "7.6.5",
  "nbformat": "5.11.1",
  "nbclient": "0.11.0",
  "ipykernel": "7.4.0",
  "setuptools": "84.2.1"
}
cpu available=True arm
mps available=True MPS float64 unsupported; explicit float32 neural execution only
cuda available=False CUDA unavailable
Local checkpoints: /Users/nicolasackermann/Documents/ChatGPT/PDEep/artifacts/training_runs
Drive checkpoints: None
Local session: no clone, install or Google Drive import/mount; local outputs only.


## Select and preview
Use `EXPERIMENT="tiny_overfit"` with `OVERFIT_UPDATES=100` first; run A-Geo and C-Geo independently. This memorizes six fixed train cases on two domains and is not generalization evidence. For the bounded experiment use `EXPERIMENT="bounded", UPDATES=1000` (or500). Validation uses paired unseen cases on familiar geometry instances. Test data stay untouched.


In [ ]:
BENCHMARK = "G1_smooth_v1"
MODELS = ("A_GEO", "C_GEO")  # requested independent medium tiny-overfit runs
CAPACITY = "medium"
PDE_FAMILIES = ("poisson", "heat", "wave")
DEVICE = "auto"  # Colab auto requires validated CUDA; local auto uses available MPS/CPU
EXPERIMENT = "tiny_overfit"  # requested100-update tiny test; bounded/smoke also available
OVERFIT_UPDATES = 100
UPDATES = 1000  # bounded budget; tiny_overfit uses OVERFIT_UPDATES automatically; smoke: <=3
ACTION = "fresh"  # fresh or resume
RESUME_CHECKPOINT = None  # verified Drive/local smooth-update-XXXXXXXX.json
SAVE_EVERY = 250
VALIDATE_EVERY = 250
RUN_MODE = "preview"  # preview, train, resume; only a fresh click launches


In [ ]:
# Optional advanced settings; defaults keep the existing architecture-specific loss.
ADVANCED = {}  # e.g. {"cases_per_geometry":4}; finite query bank is explicit
RUN_NAME = None  # unique run tag; existing fresh destinations are rejected
PERSISTENCE = SETUP["persistence"]


In [ ]:
from pdeep.smooth.workflows import prepare, preview
from pdeep.smooth.certification import load_record
from pdeep.smooth.definition import manifest
from IPython.display import display, Markdown
import json

def training_settings():
    if SETUP_CONFIGURATION != (REPO_URL, REPO_REF, PROJECT_DIR, USE_GOOGLE_DRIVE, DRIVE_ROOT, LOCAL_OUTPUT_DIR):
        raise RuntimeError("Setup changed: rerun setup and preview.")
    device = "cuda" if IS_COLAB and DEVICE == "auto" else DEVICE
    if IS_COLAB and not str(device).startswith("cuda"):
        raise RuntimeError("Colab requires validated CUDA; no silent CPU fallback.")
    return dict(models=MODELS, benchmark=BENCHMARK, capacity=CAPACITY,
        pdes=PDE_FAMILIES, device=device, experiment=EXPERIMENT, updates=OVERFIT_UPDATES if EXPERIMENT == "tiny_overfit" else UPDATES,
        action=ACTION, resume_checkpoint=RESUME_CHECKPOINT,
        save_every=SAVE_EVERY, validate_every=VALIDATE_EVERY,
        output_dir=SETUP["local_output_dir"], persistence=PERSISTENCE,
        run_name=RUN_NAME, advanced=ADVANCED)

SETTINGS = training_settings()


In [ ]:
# 4. Compatible reference certification — numerical checks only, never training.
# A valid environment certificate is reused. A new numerical stack performs a
# bounded96-case CPU64 comparison with frozen original reference values, with
# live stage progress. Interrupting this cell stops it; no worker stays running.
from pdeep.smooth.qualification import ensure_compatible_certificate
CERTIFICATE, CERTIFICATE_PATH = ensure_compatible_certificate()
print("Compatible certificate:", CERTIFICATE["certification_id"])
print("Exact canonical reference policy:", CERTIFICATE["reference_policy"])


In [ ]:
# 5. Plan review — no model/optimizer execution.
PLAN = prepare(**SETTINGS, certificate_path=CERTIFICATE_PATH)
REVIEWED_IDENTITY = PLAN.identity
print("Resolved Git commit:", SETUP["git_commit"])
print("Certification:", PLAN.certificate["certification_id"], PLAN.certificate["pass_fail_status"])
print(json.dumps(preview(PLAN), indent=2))
for config in PLAN.configs:
    print(config.architecture, "AdamW", {"learning_rate": config.learning_rate,
        "weight_decay": config.weight_decay, "scheduler": config.scheduler,
        "microbatch": config.microbatch, "accumulation": config.accumulation,
        "effective_batch": config.effective_batch, "clip_norm": config.grad_clip_norm,
        "save_every": config.checkpoint_every, "validate_every": config.validation_every,
        "weak_quadrature": (config.spatial_order, config.time_order, config.weak_tests)})
print("Sampled reference error budget (fraction of training RMS):", PLAN.certificate["numerical_error_summary"])
print("Numerical certificate covers fixed D0 polygon instances and P4 h=.02 reference; rough G1 is blocked.")
print("CPU64 reference-bank warm-up occurs on manual launch; no optimization or bank generation in preview.")
print("Checkpoint locations:", [r["local_checkpoints"] for r in preview(PLAN)])
print("Drive destination:", SETUP["drive_root"], "| optional verified local-first copy; local storage in Colab is temporary.")


## Start — one explicit click
After reviewing the preview, set `RUN_MODE="train"` (or`"resume"` with `ACTION="resume"`), rerun configuration/preview and this cell, then click. Changing any configuration requires a new preview. This is the only notebook training launch mechanism.

The first launch computes and verifies finite continuous-query banks on CPU64; it may take several minutes. Labels are point evaluations, not interpolated stored trajectory frames. Tiny-overfit uses a fixed finite pool. Bounded runs sample the full lazy12000-case/PDE train split by default; per-case point-query banks are finite. Many fresh CPU64 reference trajectories can take hours. An explicit advanced cases_per_geometry setting selects a smaller diagnostic pool and is labeled in preview.


In [ ]:
from pdeep.notebook_launch import offer_launch
from pdeep.smooth.workflows import launch
RESULTS = {}

def _start_reviewed_training():
    if training_settings() != SETTINGS:
        raise RuntimeError("Configuration changed; rerun preview and render a fresh button.")
    if RUN_MODE not in {"train", "resume"} or (RUN_MODE == "resume") != (ACTION == "resume"):
        raise RuntimeError("Choose a consistent train/fresh or resume/resume action, then review again.")
    RESULTS.update(launch(PLAN, expected_identity=REVIEWED_IDENTITY, notebook=True))

LAUNCH_BUTTON = offer_launch(_start_reviewed_training,
    description="Start reviewed A/C-Geo training", enabled=RUN_MODE in {"train", "resume"})
print("No launch from cell execution. A fresh click is required.")


In [ ]:
from pdeep.smooth.plotting import validation_table, history_plots, time_plot, representative_field
import matplotlib.pyplot as plt
if not RESULTS:
    print("Training, tiny-overfit success, and learned accuracy: pending user execution.")
for name, engine in RESULTS.items():
    print(name, "updates", engine.state.successful, "failed", engine.state.failed, "skipped", engine.state.skipped)
    print("Last local checkpoint:", engine.state.last_checkpoint)
    print("Local save / Drive sync:", engine.state.persistence)
    print("Histories and assessments:", engine.run_dir)
    if engine.config.preset == "tiny_overfit": print(engine.overfit_assessment())
    if engine.latest_validation:
        display(Markdown(validation_table(engine.latest_validation)))
        time_plot(engine.latest_validation)
    history_plots(engine.run_dir)
    representative_field(engine, pde="wave")
plt.show()


## Resume and evaluate
After disconnect: reopen this notebook from GitHub at the **same commit/tag**, run both setup cells and remount the same Drive root. Select the matching model/capacity/PDEs, set`ACTION="resume"`, choose the verified Drive checkpoint manifest, preview, set`RUN_MODE="resume"`, and click. The saved budget, optimizer moments, scheduler, RNG, ordering and counters are restored; the budget is not silently extended. Missing local query banks regenerate from the saved deterministic recipe.

Local checkpoints: `SETUP["local_output_dir"]/G1_smooth_v1/runs/<architecture>/<run_id>/checkpoints/`. Verified Drive copies: `DRIVE_ROOT/checkpoints/<run_id>/<artifact_id>/smooth-update-XXXXXXXX.json`. A sync failure leaves the local checkpoint intact and emits a warning.

Open **03_evaluate.ipynb**, select a smooth checkpoint in its G1_smooth section, and explicitly enable seen-geometry evaluation. Geometry-held-out evaluation is a separate opt-in action; remeshing is never called new-geometry generalization.

Advanced: original T1 A/B0/B1/C training APIs remain in`pdeep.notebook_workflows`; the archival research notebook and all old checkpoints remain intact. This notebook defaults to the certified smooth workflow.
